In [23]:
%%writefile economia.py

# ==========================================
# ECONOMIA V10 (Com Setor Informal)
# ==========================================

from demografia import forca_trabalho


# ==========================================\n
# PRODUTIVIDADE
# ==========================================\n

def calcular_produtividade(
    educacao,
    inovacao,
    capital
):
    produtividade = (
        1 +
        educacao / 120 +
        inovacao / 150 +
        capital / 100000
    )
    return max(0.5, produtividade)


# ==========================================\n
# CONFIANÇA ECONÔMICA
# ==========================================\n

def confianca_economica(ilha):
    confianca = (
        ilha.liberdade * 0.40 +
        (100 - ilha.desemprego) / 100 * 0.30 +
        ilha.educacao / 100 * 0.15 +
        ilha.inovacao / 100 * 0.15
    )
    return max(0.10, min(1.50, confianca))


# ==========================================\n
# EMPRESAS
# ==========================================\n

def atualizar_empresas(ilha):
    confianca = confianca_economica(ilha)

    novas = int(
        ilha.liberdade *
        confianca *
        ilha.inovacao *
        0.05
    )

    falencias = int(
        ilha.juros / 4 +
        ilha.desemprego / 20
    )

    ilha.empresas += (novas - falencias)
    ilha.empresas = max(10, ilha.empresas)

    return ilha


# ==========================================\n
# EMPREGO FORMAL E INFORMAL (ESTILO BRASIL)
# ==========================================\n

def gerar_empregos(ilha):
    trabalhadores = forca_trabalho(ilha)

    # Vagas formais geradas pela capacidade econômica
    vagas_formais = int(
        (ilha.capital * ilha.liberdade) / 60
    )

    ilha.empregados = min(trabalhadores, vagas_formais)

    restante_sem_vaga = trabalhadores - ilha.empregados

    # Uma parte dos sem-vaga vai para a informalidade (bicos / conta própria)
    # No Brasil, a informalidade absorve grande parte do excedente
    ilha.informais = int(restante_sem_vaga * 0.70)

    # O restante real fica totalmente desempregado
    ilha.desempregados = max(0, restante_sem_vaga - ilha.informais)

    # Taxa de desemprego oficial considera apenas quem não tem absolutamente nada nem informal
    ilha.desemprego = (
        ilha.desempregados / max(1, trabalhadores)
    ) * 100

    return ilha


# ==========================================\n
# PRODUÇÃO
# ==========================================\n

def atualizar_produtividade(ilha):
    ilha.produtividade = calcular_produtividade(
        ilha.educacao,
        ilha.inovacao,
        ilha.capital
    )
    return ilha


# ==========================================\n
# SALÁRIOS E LUCROS (Considerando rendas formais e informais)
# ==========================================\n

def distribuir_renda(ilha):
    distribuicao = {
        "Socialismo": 0.85,
        "Social Democracia": 0.75,
        "Conservador": 0.65,
        "Liberal": 0.55
    }

    percentual_salario = distribuicao[ilha.nome]

    # Massa salarial formal + estimativa menor gerada pelos informais
    massa_formal = ilha.pib * percentual_salario
    massa_informal = ilha.informais * 0.4  # Informais ganham menos

    massa_salarial_total = massa_formal + massa_informal
    lucro_total = max(0, ilha.pib - massa_salarial_total)

    ilha.massa_salarial = massa_salarial_total
    ilha.lucro_total = lucro_total

    total_trabalhadores_com_renda = max(1, ilha.empregados + ilha.informais)
    ilha.salario_medio = massa_salarial_total / total_trabalhadores_com_renda

    return ilha


# ==========================================\n
# CAPITAL
# ==========================================\n

def acumular_capital(ilha):
    reinvestimento = (
        ilha.lucro_total *
        0.25 *
        (1 - ilha.juros / 100)
    )
    reinvestimento = max(0, reinvestimento)
    ilha.capital += reinvestimento
    return ilha


# ==========================================\n
# PIB PER CAPITA
# ==========================================\n

def pib_per_capita(ilha):
    return ilha.pib / max(1, ilha.populacao)


# ==========================================\n
# ATUALIZAÇÃO ECONÔMICA
# ==========================================\n

def atualizar_economia(ilha):
    atualizar_empresas(ilha)
    gerar_empregos(ilha)
    atualizar_produtividade(ilha)

    # PIB impulsionado por formais e parcialmente pelos informais
    trabalho_efetivo = ilha.empregados + (ilha.informais * 0.5)
    ilha.pib = trabalho_efetivo * ilha.produtividade

    distribuir_renda(ilha)
    acumular_capital(ilha)

    return ilha

Overwriting economia.py


In [12]:
%%writefile indicadores.py

import numpy as np


# ==========================================
# GINI
# ==========================================

def calcular_gini(rendas):

    rendas = np.array(
        [max(0.01, r) for r in rendas]
    )

    rendas = np.sort(rendas)

    n = len(rendas)

    acumulado = np.cumsum(rendas)

    if acumulado[-1] == 0:
        return 0

    gini = (
        n + 1
        - 2 * np.sum(acumulado) / acumulado[-1]
    ) / n

    return max(
        0,
        min(1, gini)
    )


# ==========================================
# EXPECTATIVA DE VIDA
# ==========================================

def calcular_expectativa_vida(saude):

    expectativa = (

        50 +

        saude * 0.35

    )

    return max(
        40,
        min(90, expectativa)
    )


# ==========================================
# FELICIDADE
# ==========================================

def calcular_felicidade(

    saude,
    educacao,
    desemprego,
    fome_grave,
    corrupcao,
    expectativa_vida

):

    felicidade = (

        saude * 0.20 +

        educacao * 0.15 +

        (100 - desemprego) * 0.20 +

        (100 - fome_grave * 100) * 0.20 +

        (100 - corrupcao * 100) * 0.10 +

        expectativa_vida * 0.15

    )

    return max(
        0,
        min(100, felicidade)
    )


# ==========================================
# ATRATIVIDADE MIGRATÓRIA
# ==========================================

def calcular_atratividade(

    felicidade,
    salario_medio,
    desemprego,
    expectativa_vida

):

    score = (

        felicidade * 0.40 +

        min(100, salario_medio) * 0.30 +

        (100 - desemprego) * 0.20 +

        expectativa_vida * 0.10

    )

    return score


# ==========================================
# SEGURANÇA ALIMENTAR
# ==========================================

def atualizar_seguranca_alimentar(

    inseguranca_leve,
    inseguranca_moderada,
    inseguranca_grave,
    saldo_alimentos

):

    if saldo_alimentos > 0:

        inseguranca_grave = max(
            0,
            inseguranca_grave - 0.002
        )

        inseguranca_moderada = max(
            0,
            inseguranca_moderada - 0.001
        )

    else:

        inseguranca_grave = min(
            0.30,
            inseguranca_grave + 0.003
        )

        inseguranca_moderada = min(
            0.40,
            inseguranca_moderada + 0.002
        )

    seguranca_alimentar = max(

        0,

        1
        - inseguranca_leve
        - inseguranca_moderada
        - inseguranca_grave

    )

    return (

        seguranca_alimentar,

        inseguranca_leve,

        inseguranca_moderada,

        inseguranca_grave

    )

Writing indicadores.py


In [13]:
%%writefile governo.py

# ==========================================
# GOVERNO V9
# ==========================================

# Corrupção institucional
# Finanças públicas
# Juros
# Arrecadação
# Gastos
# Dívida pública


# ==========================================
# PARÂMETROS INSTITUCIONAIS
# ==========================================

def parametros_institucionais(nome_regime):

    # Estes números não representam países reais.
    # São apenas parâmetros da simulação.

    configuracao = {

        "Socialismo": {

            "transparencia": 0.70,
            "fiscalizacao": 0.75,
            "concentracao_poder": 0.70

        },

        "Social Democracia": {

            "transparencia": 0.85,
            "fiscalizacao": 0.85,
            "concentracao_poder": 0.35

        },

        "Conservador": {

            "transparencia": 0.75,
            "fiscalizacao": 0.70,
            "concentracao_poder": 0.50

        },

        "Liberal": {

            "transparencia": 0.80,
            "fiscalizacao": 0.65,
            "concentracao_poder": 0.40

        }
    }

    return configuracao[nome_regime]


# ==========================================
# CORRUPÇÃO
# ==========================================

def calcular_corrupcao(

    transparencia,
    fiscalizacao,
    concentracao_poder

):

    corrupcao = (

        concentracao_poder * 0.40 +

        (1 - transparencia) * 0.30 +

        (1 - fiscalizacao) * 0.30

    )

    return max(
        0.02,
        min(0.40, corrupcao)
    )


# ==========================================
# TAXA DE JUROS
# ==========================================

def calcular_juros(

    divida,
    pib

):

    if pib <= 0:

        return 10.0

    relacao = divida / pib

    juros = (

        4 +

        relacao * 2

    )

    return max(
        2,
        min(20, juros)
    )


# ==========================================
# ARRECADAÇÃO
# ==========================================

def arrecadar_impostos(

    pib,
    imposto,
    corrupcao

):

    arrecadacao_bruta = (

        pib * imposto

    )

    perda_corrupcao = (

        arrecadacao_bruta *

        corrupcao

    )

    arrecadacao_liquida = (

        arrecadacao_bruta -

        perda_corrupcao

    )

    return {

        "bruta": arrecadacao_bruta,

        "perdida": perda_corrupcao,

        "liquida": arrecadacao_liquida

    }


# ==========================================
# GASTO SOCIAL
# ==========================================

def calcular_gastos(

    arrecadacao,

    gasto_social,

    investimento_publico,

    idosos

):

    gasto_saude_educacao = (

        arrecadacao *

        (
            gasto_social +

            investimento_publico
        )

    )

    previdencia = (

        idosos * 2

    )

    gasto_total = (

        gasto_saude_educacao +

        previdencia

    )

    return {

        "total": gasto_total,

        "previdencia": previdencia

    }


# ==========================================
# DÍVIDA
# ==========================================

def atualizar_divida(

    divida,

    arrecadacao,

    gastos

):

    saldo = (

        arrecadacao -

        gastos

    )

    if saldo < 0:

        divida += abs(saldo)

    else:

        divida = max(
            0,
            divida - saldo
        )

    return divida


# ==========================================
# ATUALIZAÇÃO GOVERNAMENTAL
# ==========================================

def atualizar_governo(ilha):

    inst = parametros_institucionais(
        ilha.nome
    )

    ilha.transparencia = (
        inst["transparencia"]
    )

    ilha.fiscalizacao = (
        inst["fiscalizacao"]
    )

    ilha.concentracao_poder = (
        inst["concentracao_poder"]
    )

    ilha.corrupcao = (

        calcular_corrupcao(

            ilha.transparencia,

            ilha.fiscalizacao,

            ilha.concentracao_poder

        )

    )

    arrec = arrecadar_impostos(

        ilha.pib,

        ilha.imposto,

        ilha.corrupcao

    )

    ilha.arrecadacao = (
        arrec["liquida"]
    )

    ilha.perda_corrupcao = (
        arrec["perdida"]
    )

    gastos = calcular_gastos(

        arrec["liquida"],

        ilha.gasto_social,

        ilha.investimento,

        ilha.idosos

    )

    ilha.gasto_publico = (
        gastos["total"]
    )

    ilha.previdencia = (
        gastos["previdencia"]
    )

    ilha.divida = (

        atualizar_divida(

            ilha.divida,

            ilha.arrecadacao,

            ilha.gasto_publico

        )

    )

    ilha.juros = (

        calcular_juros(

            ilha.divida,

            ilha.pib

        )

    )

    return ilha

Writing governo.py


In [14]:
%%writefile recursos.py

# ==========================================
# RECURSOS V9
# ==========================================

def fator_recursos(ilha):

    energia_ratio = min(
        1.0,
        ilha.energia / 50000
    )

    minerio_ratio = min(
        1.0,
        ilha.minerio / 50000
    )

    fator = min(
        energia_ratio,
        minerio_ratio
    )

    return max(
        0.25,
        fator
    )


# ==========================================
# PRODUÇÃO AGRÍCOLA
# ==========================================

def produzir_alimentos(ilha):

    producao = (

        ilha.empregados *

        ilha.produtividade *

        ilha.incentivo *

        0.45

    )

    ilha.producao_agricola = producao

    return producao


# ==========================================
# PRODUÇÃO INDUSTRIAL
# ==========================================

def produzir_industria(ilha):

    fator = fator_recursos(
        ilha
    )

    producao = (

        ilha.empregados *

        ilha.produtividade *

        fator *

        0.35

    )

    ilha.producao_industrial = (
        producao
    )

    return producao


# ==========================================
# SERVIÇOS
# ==========================================

def produzir_servicos(ilha):

    producao = (

        ilha.empregados *

        ilha.produtividade *

        0.20

    )

    ilha.producao_servicos = (
        producao
    )

    return producao


# ==========================================
# CONSUMO DE ALIMENTOS
# ==========================================

def atualizar_alimentos(ilha):

    consumo = (

        ilha.populacao * 0.9

    )

    ilha.alimentos += (
        ilha.producao_agricola
    )

    ilha.alimentos -= consumo

    ilha.alimentos = max(
        0,
        ilha.alimentos
    )

    return ilha


# ==========================================
# CONSUMO DE ENERGIA
# ==========================================

def consumir_energia(ilha):

    consumo = (

        ilha.producao_industrial *

        0.08

    )

    ilha.energia -= consumo

    ilha.energia = max(
        0,
        ilha.energia
    )

    return ilha


# ==========================================
# CONSUMO DE MINÉRIO
# ==========================================

def consumir_minerio(ilha):

    consumo = (

        ilha.producao_industrial *

        0.05

    )

    ilha.minerio -= consumo

    ilha.minerio = max(
        0,
        ilha.minerio
    )

    return ilha


# ==========================================
# REGENERAÇÃO
# ==========================================

def regenerar_recursos(ilha):

    ilha.energia += (

        500 +

        ilha.inovacao * 10

    )

    ilha.minerio += (

        100 +

        ilha.inovacao * 2

    )

    ilha.energia = min(
        200000,
        ilha.energia
    )

    ilha.minerio = min(
        200000,
        ilha.minerio
    )

    return ilha


# ==========================================
# CICLO COMPLETO
# ==========================================

def atualizar_recursos(ilha):

    produzir_alimentos(
        ilha
    )

    produzir_industria(
        ilha
    )

    produzir_servicos(
        ilha
    )

    atualizar_alimentos(
        ilha
    )

    consumir_energia(
        ilha
    )

    consumir_minerio(
        ilha
    )

    regenerar_recursos(
        ilha
    )

    return ilha

Writing recursos.py


In [15]:
%%writefile comercio.py

# ==========================================
# COMERCIO V9
# ==========================================

#
# Sistema de comércio entre ilhas
#
# Recursos:
#
# - alimentos
# - energia
# - minério
#
# Uma ilha pode vender excedentes
# para outra com escassez.
#
# ==========================================


# ==========================================
# CONSTANTES
# ==========================================

EXPORT_ALIMENTOS = 20000

EXPORT_ENERGIA = 10000

EXPORT_MINERIO = 5000

VALOR_ALIMENTOS = 5000

VALOR_ENERGIA = 4000

VALOR_MINERIO = 3500


# ==========================================
# COMÉRCIO ALIMENTAR
# ==========================================

def comerciar_alimentos(

    exportadora,
    importadora

):

    if (

        exportadora.alimentos > 300000

        and

        importadora.alimentos < 100000

    ):

        exportadora.alimentos -= (
            EXPORT_ALIMENTOS
        )

        importadora.alimentos += (
            EXPORT_ALIMENTOS
        )

        exportadora.pib += (
            VALOR_ALIMENTOS
        )

        exportadora.exportacoes += 1

        importadora.importacoes += 1

        return True

    return False


# ==========================================
# COMÉRCIO DE ENERGIA
# ==========================================

def comerciar_energia(

    exportadora,
    importadora

):

    if (

        exportadora.energia > 120000

        and

        importadora.energia < 30000

    ):

        exportadora.energia -= (
            EXPORT_ENERGIA
        )

        importadora.energia += (
            EXPORT_ENERGIA
        )

        exportadora.pib += (
            VALOR_ENERGIA
        )

        exportadora.exportacoes += 1

        importadora.importacoes += 1

        return True

    return False


# ==========================================
# COMÉRCIO DE MINÉRIO
# ==========================================

def comerciar_minerio(

    exportadora,
    importadora

):

    if (

        exportadora.minerio > 120000

        and

        importadora.minerio < 30000

    ):

        exportadora.minerio -= (
            EXPORT_MINERIO
        )

        importadora.minerio += (
            EXPORT_MINERIO
        )

        exportadora.pib += (
            VALOR_MINERIO
        )

        exportadora.exportacoes += 1

        importadora.importacoes += 1

        return True

    return False


# ==========================================
# MELHOR EXPORTADOR
# ==========================================

def ilha_com_mais(

    ilhas,
    atributo

):

    return max(
        ilhas,
        key=lambda x: getattr(
            x,
            atributo
        )
    )


# ==========================================
# MAIOR NECESSIDADE
# ==========================================

def ilha_com_menos(

    ilhas,
    atributo

):

    return min(
        ilhas,
        key=lambda x: getattr(
            x,
            atributo
        )
    )


# ==========================================
# EXECUTA COMÉRCIO GLOBAL
# ==========================================

def atualizar_comercio(

    ilhas

):

    # ------------------
    # ALIMENTOS
    # ------------------

    rica = ilha_com_mais(
        ilhas,
        "alimentos"
    )

    pobre = ilha_com_menos(
        ilhas,
        "alimentos"
    )

    if rica != pobre:

        comerciar_alimentos(
            rica,
            pobre
        )

    # ------------------
    # ENERGIA
    # ------------------

    rica = ilha_com_mais(
        ilhas,
        "energia"
    )

    pobre = ilha_com_menos(
        ilhas,
        "energia"
    )

    if rica != pobre:

        comerciar_energia(
            rica,
            pobre
        )

    # ------------------
    # MINÉRIO
    # ------------------

    rica = ilha_com_mais(
        ilhas,
        "minerio"
    )

    pobre = ilha_com_menos(
        ilhas,
        "minerio"
    )

    if rica != pobre:

        comerciar_minerio(
            rica,
            pobre
        )

    return ilhas


# ==========================================
# BALANÇA COMERCIAL
# ==========================================

def saldo_comercial(

    ilha

):

    return (

        ilha.exportacoes -

        ilha.importacoes

    )

Writing comercio.py


In [29]:
%%writefile main.py

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

from config import *

from demografia import (
    atualizar_demografia
)

from economia import (
    atualizar_economia,
    distribuir_renda,
    pib_per_capita
)

from governo import (
    atualizar_governo
)

from recursos import (
    atualizar_recursos
)

from comercio import (
    atualizar_comercio,
    saldo_comercial
)

from migracao import (
    atualizar_migracao,
    saldo_migratorio
)

from indicadores import (
    calcular_felicidade,
    atualizar_seguranca_alimentar
)


# ==========================================
# CLASSE ILHA
# ==========================================

class Ilha:

    def __init__(
        self,
        nome,
        imposto,
        liberdade,
        gasto_social,
        investimento,
        incentivo
    ):

        self.nome = nome

        # -------------------------\n
        # DEMOGRAFIA
        # -------------------------\n

        self.criancas = CRIANCAS_INICIAL
        self.adultos = ADULTOS_INICIAL
        self.idosos = IDOSOS_INICIAL

        self.total_nascimentos = 0
        self.total_mortes = 0

        self.imigrantes = 0
        self.emigrantes = 0

        # -------------------------\n
        # ECONOMIA
        # -------------------------\n

        self.capital = CAPITAL_INICIAL

        self.empresas = EMPRESAS_INICIAIS

        self.empregados = 0
        self.desempregados = 0
        self.desemprego = 0

        self.produtividade = 1

        self.salario_medio = 0

        self.massa_salarial = 0
        self.lucro_total = 0

        self.pib = 0

        # -------------------------\n
        # RECURSOS
        # -------------------------\n

        self.alimentos = ALIMENTOS_INICIAIS
        self.energia = ENERGIA_INICIAL
        self.minerio = MINERIO_INICIAL

        self.producao_agricola = 0
        self.producao_industrial = 0
        self.producao_servicos = 0

        # -------------------------\n
        # SOCIAL
        # -------------------------\n

        self.educacao = EDUCACAO_INICIAL
        self.saude = SAUDE_INICIAL
        self.inovacao = INOVACAO_INICIAL

        self.expectativa_vida = 70

        self.felicidade = FELICIDADE_INICIAL

        # -------------------------\n
        # SEGURANÇA ALIMENTAR
        # -------------------------\n

        self.inseguranca_grave = (
            INSEGURANCA_GRAVE_INICIAL
        )

        self.inseguranca_moderada = (
            INSEGURANCA_MODERADA_INICIAL
        )

        self.inseguranca_leve = (
            INSEGURANCA_LEVE_INICIAL
        )

        self.seguranca_alimentar = 1 - (
            self.inseguranca_grave +
            self.inseguranca_moderada +
            self.inseguranca_leve
        )

        # -------------------------\n
        # GOVERNO
        # -------------------------\n

        self.corrupcao = 0

        self.transparencia = 0
        self.fiscalizacao = 0
        self.concentracao_poder = 0

        self.arrecadacao = 0
        self.gasto_publico = 0

        self.previdencia = 0

        self.perda_corrupcao = 0

        self.divida = 0

        self.juros = JUROS_BASE

        # -------------------------\n
        # POLÍTICAS
        # -------------------------\n

        self.imposto = imposto

        self.liberdade = liberdade

        self.gasto_social = gasto_social

        self.investimento = investimento

        self.incentivo = incentivo

        # -------------------------\n
        # COMÉRCIO
        # -------------------------\n

        self.exportacoes = 0
        self.importacoes = 0

    @property
    def populacao(self):

        return (

            self.criancas +

            self.adultos +

            self.idosos

        )


# ==========================================\n
# REGIMES
# ==========================================\n

ilhas = [

    Ilha(
        "Socialismo",
        0.60,
        0.20,
        0.90,
        0.90,
        0.70
    ),

    Ilha(
        "Social Democracia",
        0.40,
        0.60,
        0.80,
        0.70,
        0.85
    ),

    Ilha(
        "Conservador",
        0.30,
        0.70,
        0.50,
        0.50,
        0.95
    ),

    Ilha(
        "Liberal",
        0.15,
        0.95,
        0.20,
        0.20,
        1.10
    )

]


# ==========================================\n
# ENTRADA DA QUANTIDADE DE ANOS
# ==========================================\n

try:
    entrada = input(f"Digite a quantidade de anos de simulação [Padrão: {ANOS_SIMULACAO}]: ")
    ANOS_SIMULACAO_USER = int(entrada) if entrada.strip() != "" else ANOS_SIMULACAO
except (KeyboardInterrupt, EOFError):
    ANOS_SIMULACAO_USER = ANOS_SIMULACAO
    print(f"\nUsando o valor padrão de {ANOS_SIMULACAO} anos.")


# ==========================================\n
# HISTÓRICO
# ==========================================\n

historico = []


# ==========================================\n
# SIMULAÇÃO
# ==========================================\n

for ano in range(ANOS_SIMULACAO_USER):

    for ilha in ilhas:

        atualizar_demografia(ilha)

        atualizar_economia(ilha)

        atualizar_recursos(ilha)

        # PIB real gerado pelos setores

        ilha.pib = (

            ilha.producao_agricola +

            ilha.producao_industrial +

            ilha.producao_servicos

        )

        distribuir_renda(ilha)

        atualizar_governo(ilha)

        saldo_alimentos = (

            ilha.alimentos -

            ilha.populacao

        )

        (
            ilha.seguranca_alimentar,
            ilha.inseguranca_leve,
            ilha.inseguranca_moderada,
            ilha.inseguranca_grave

        ) = atualizar_seguranca_alimentar(

            ilha.inseguranca_leve,
            ilha.inseguranca_moderada,
            ilha.inseguranca_grave,
            saldo_alimentos

        )

        ilha.felicidade = (

            calcular_felicidade(

                ilha.saude,

                ilha.educacao,

                ilha.desemprego,

                ilha.inseguranca_grave,

                ilha.corrupcao,

                ilha.expectativa_vida

            )

        )

    atualizar_comercio(ilhas)

    atualizar_migracao(ilhas)

    historico.append({

        "Ano": ano,

        **{
            f"PIB_{i.nome}": i.pib
            for i in ilhas
        }

    })


# ==========================================\n
# RESULTADO FINAL
# ==========================================\n

resultado = []

for ilha in ilhas:

    resultado.append({

        "Modelo": ilha.nome,

        "Populacao": int(
            ilha.populacao
        ),

        "PIB": round(
            ilha.pib
        ),

        "PIB_per_capita": round(
            pib_per_capita(ilha),
            2
        ),

        "Capital": round(
            ilha.capital
        ),

        "Empresas": ilha.empresas,

        "Empregados": ilha.empregados,

        "Desemprego_%": round(
            ilha.desemprego,
            2
        ),

        "Salario_Medio": round(
            ilha.salario_medio,
            2
        ),

        "Produtividade": round(
            ilha.produtividade,
            2
        ),

        "Nascimentos": int(
            ilha.total_nascimentos
        ),

        "Mortes": int(
            ilha.total_mortes
        ),

        "Imigrantes": ilha.imigrantes,

        "Emigrantes": ilha.emigrantes,

        "Saldo_Migratorio":
            saldo_migratorio(ilha),

        "Exportacoes":
            ilha.exportacoes,

        "Importacoes":
            ilha.importacoes,

        "Saldo_Comercial":
            saldo_comercial(ilha),

        "Alimentos":
            round(ilha.alimentos),

        "Energia":
            round(ilha.energia),

        "Minerio":
            round(ilha.minerio),

        "Prod_Agro":
            round(
                ilha.producao_agricola
            ),

        "Prod_Industria":
            round(
                ilha.producao_industrial
            ),

        "Prod_Servicos":
            round(
                ilha.producao_servicos
            ),

        "Fome_Grave_%":
            round(
                ilha.inseguranca_grave * 100,
                2
            ),

        "Pessoas_em_Fome":
            int(
                ilha.populacao *
                ilha.inseguranca_grave
            ),

        "Expectativa_Vida":
            round(
                ilha.expectativa_vida,
                1
            ),

        "Corrupcao_%":
            round(
                ilha.corrupcao * 100,
                2
            ),

        "Transparencia":
            round(
                ilha.transparencia,
                2
            ),

        "Fiscalizacao":
            round(
                ilha.fiscalizacao,
                2
            ),

        "Perda_Corrupcao":
            round(
                ilha.perda_corrupcao
            ),

        "Juros_%":
            round(
                ilha.juros,
                2
            ),

        "Divida":
            round(
                ilha.divida
            ),

        "Felicidade":
            round(
                ilha.felicidade,
                2
            )

    })

resultado = pd.DataFrame(resultado)

print(f"\n=== RESULTADO FINAL V9 ({ANOS_SIMULACAO_USER} ANOS) ===\n")

print(
    resultado.sort_values(
        by="Felicidade",
        ascending=False
    )
)

# ==========================================\n
# GRÁFICO
# ==========================================\n

hist = pd.DataFrame(historico)

plt.figure(figsize=(12,6))

for ilha in ilhas:

    plt.plot(
        hist["Ano"],
        hist[f"PIB_{ilha.nome}"],
        label=ilha.nome
    )

plt.title(f"Evolução do PIB ({ANOS_SIMULACAO_USER} Anos)")
plt.xlabel("Ano")
plt.ylabel("PIB")
plt.legend()
plt.grid(True)

plt.show()

Overwriting main.py


In [3]:
%%writefile config.py

# ==========================================
# CONFIGURAÇÕES GERAIS
# ==========================================

ANOS_SIMULACAO = 100

# ==========================================
# POPULAÇÃO INICIAL
# ==========================================

POPULACAO_INICIAL = 10000

CRIANCAS_INICIAL = 2500
ADULTOS_INICIAL = 6000
IDOSOS_INICIAL = 1500

# ==========================================
# SEGURANÇA ALIMENTAR
# ==========================================

# Inspirado na EBIA / dados brasileiros recentes

INSEGURANCA_GRAVE_INICIAL = 0.032
INSEGURANCA_MODERADA_INICIAL = 0.068
INSEGURANCA_LEVE_INICIAL = 0.20

# ==========================================
# RECURSOS INICIAIS
# ==========================================

ALIMENTOS_INICIAIS = 250000

ENERGIA_INICIAL = 100000

MINERIO_INICIAL = 100000

CAPITAL_INICIAL = 10000

# ==========================================
# EMPRESAS
# ==========================================

EMPRESAS_INICIAIS = 100

# ==========================================
# INDICADORES SOCIAIS
# ==========================================

EDUCACAO_INICIAL = 50

SAUDE_INICIAL = 48

INOVACAO_INICIAL = 50

FELICIDADE_INICIAL = 58

# ==========================================
# EVENTOS ECONÔMICOS
# ==========================================

PROB_BOOM = 0.03

PROB_RECESSAO = 0.05

# ==========================================
# SISTEMA FINANCEIRO
# ==========================================

JUROS_BASE = 5.0

CREDITO_BASE = 10000

# ==========================================
# COMÉRCIO
# ==========================================

LIMITE_EXPORTACAO_ALIMENTOS = 300000

LIMITE_IMPORTACAO_ALIMENTOS = 100000

VOLUME_COMERCIO_ALIMENTOS = 20000

VOLUME_COMERCIO_ENERGIA = 10000

VOLUME_COMERCIO_MINERIO = 10000

Writing config.py


In [17]:
!ls

comercio.py    economia.py     main.py	    recursos.py
config.py      governo.py      migracao.py  sample_data
demografia.py  indicadores.py  __pycache__


In [22]:
%%writefile demografia.py

# ==========================================
# DEMOGRAFIA V10 (Com Informalidade)
# ==========================================

from indicadores import calcular_expectativa_vida


# ==========================================
# NATALIDADE
# ==========================================

def calcular_natalidade(
    adultos,
    educacao,
    saude
):

    taxa = (
        0.024
        - educacao / 12000
        + saude / 50000
    )

    taxa = max(
        0.008,
        min(0.04, taxa)
    )

    nascimentos = adultos * taxa

    return nascimentos


# ==========================================
# MORTALIDADE
# ==========================================

def calcular_mortalidade(
    criancas,
    adultos,
    idosos,
    saude,
    fome_grave
):

    # mortalidade infantil
    mortes_criancas = criancas * (
        0.0025
        + fome_grave * 0.002
    )

    # adultos
    mortes_adultos = adultos * (
        0.005
        + fome_grave * 0.003
    )

    # idosos
    fator_saude_idosos = max(0.005, 0.018 - (saude / 5000))
    mortes_idosos = idosos * fator_saude_idosos

    return int(mortes_criancas), int(mortes_adultos), int(mortes_idosos)


# ==========================================
# FORÇA DE TRABALHO TOTAL
# ==========================================

def forca_trabalho(ilha):
    return int(ilha.adultos * 0.85)


# ==========================================
# ATUALIZAÇÃO DEMOGRÁFICA
# ==========================================

def atualizar_demografia(ilha):

    ilha.expectativa_vida = calcular_expectativa_vida(ilha.saude)

    nascimentos = int(calcular_natalidade(
        ilha.adultos,
        ilha.educacao,
        ilha.saude
    ))

    m_criancas, m_adultos, m_idosos = calcular_mortalidade(
        ilha.criancas,
        ilha.adultos,
        ilha.idosos,
        ilha.saude,
        ilha.inseguranca_grave
    )

    total_mortes = m_criancas + m_adultos + m_idosos

    transicao_criancas = int(ilha.criancas * 0.05)
    transicao_adultos = int(ilha.adultos * 0.03)

    ilha.criancas = max(0, ilha.criancas + nascimentos - m_criancas - transicao_criancas)
    ilha.adultos = max(0, ilha.adultos + transicao_criancas - m_adultos - transicao_adultos)
    ilha.idosos = max(0, ilha.idosos + transicao_adultos - m_idosos)

    ilha.total_nascimentos += nascimentos
    ilha.total_mortes += total_mortes

    return ilha

Overwriting demografia.py


In [36]:
%%writefile migracao.py

# ==========================================
# MIGRAÇÃO V11 (Forçada e Direta)
# ==========================================

def atualizar_migracao(ilhas):
    # Encontra a ilha com maior felicidade e a com menor felicidade
    mais_feliz = max(ilhas, key=lambda x: x.felicidade)
    menos_feliz = min(ilhas, key=lambda x: x.felicidade)

    if mais_feliz != menos_feliz:
        # Se houver diferença de felicidade, há fluxo migratório
        diferenca = mais_feliz.felicidade - menos_feliz.felicidade

        if diferenca > 0.5:
            # 2% da população adulta da ilha menos feliz migra para a mais feliz
            migrantes = int(menos_feliz.adultos * 0.02)

            if migrantes > 0 and menos_feliz.adultos > migrantes:
                menos_feliz.adultos -= migrantes
                mais_feliz.adultos += migrantes

                menos_feliz.emigrantes += migrantes
                mais_feliz.imigrantes += migrantes

    return ilhas

def saldo_migratorio(ilha):
    return ilha.imigrantes - ilha.emigrantes

def resumo_migracao(ilha):
    return {
        "imigrantes": ilha.imigrantes,
        "emigrantes": ilha.emigrantes,
        "saldo": saldo_migratorio(ilha)
    }

Writing migracao.py


In [37]:
!python main.py

Digite a quantidade de anos de simulação [Padrão: 100]: 

=== RESULTADO FINAL V9 (100 ANOS) ===

              Modelo  Populacao  PIB  PIB_per_capita  Capital  Empresas  Empregados  Desemprego_%  Salario_Medio  Produtividade  Nascimentos  Mortes  Imigrantes  Emigrantes  Saldo_Migratorio  Exportacoes  Importacoes  Saldo_Comercial  Alimentos  Energia  Minerio  Prod_Agro  Prod_Industria  Prod_Servicos  Fome_Grave_%  Pessoas_em_Fome  Expectativa_Vida  Corrupcao_%  Transparencia  Fiscalizacao  Perda_Corrupcao  Juros_%   Divida  Felicidade
1  Social Democracia      15426  173            0.01    10000        10         100         28.98           0.44           1.85        10391    9193        4228           0              4228            0            0                0          0   199482   119676         71              65             37          23.4             3609              66.8         23.0           0.85          0.85               16       20  1478785       64.34
3            Libe